# Conditional all-labeled final fit and CSV for V011

Default execution defines helpers only: no assignment-data reads, validation-outcome reads, model loads/fits, inference, filesystem writes, external access, or submission. The standing controlled-iteration approval does not waive the frozen selection and confirmation gates.

This notebook consumes the **single immutable lowest-A qualifying choice** in `outputs/confirmation_v011/locked_before_D.json` and a completed `confirmation_result.json`. It requires the locked-selection compact canonical SHA256, the independently reviewed raw D-result SHA256, and this notebook's own frozen code SHA256 as external run inputs. It verifies all recorded confirmation checkpoint/report/prediction/row-ID/ensemble hashes and the fixed weighted predictions without reading D labels. D must show strictly positive native point-weighted gain and a wholly negative whole-breath bootstrap interval. No runner-up, weight search or changed recipe is allowed after confirmation failure. Fresh grouped D is resampling evidence within original Afit, not an untouched test.

The five completed fixed A candidates include V012's predeclared deep.5 plus original compact, curve, memory and offset each.125. The final gate compares all five before accepting the single lowest qualifying frozen choice.

Only the selected new members are fitted on all **67,905 approved labeled breaths**, using fresh all-training-only statistics, the original seed20261102, batch256, Adam/no decay, original scalar L1 or the frozen curve auxiliary, and the exact A-selected epoch/rate prefix via `fit_fixed_v008`. No holdout arrays or labels are supplied to the final fitter. Exact optimizer/RNG/checkpoint resumption remains enforced by the unchanged shared trainer. A shared `outputs/.v011_gpu.lock` prevents overlap with confirmation work.

The accepted original compact193 and deep189 all-training checkpoints are **reused**, only after exact accepted checkpoint, manifest, source record/report and preserved CSV hashes; data/feature/target fingerprints; recipe/rate-prefix/count/group checks; full independently refitted preprocessing comparison; and strict model-state loading. Their exact accepted hashes are embedded here. They are never overwritten or silently retrained. New architecture members retain their already-frozen execution backend and source fingerprints. This notebook does not select a candidate while V009 A screening is still in progress.

Inference uses the original known-input combined47 feature code on the approved test inputs. All603,600 test/sample IDs must match exactly in order. Every member prediction and the fixed weighted native float64 predictions are retained with hashes. The CSV round-trips every ID and prediction exactly, contains all phases/rows, and is written only to `outputs/confirmation_v011/final_all/submission.csv`. The accepted .18550 CSV is checked before and after work and preserved. No clipping, snapping, target-based prediction feature, or submission call is present.

The final manifest records source/configuration/data/group/row/preprocessing/checkpoint/member-prediction/weighted-prediction/CSV hashes, resumption identities, reuse provenance, runtime, and the immutable D pass. All model implementations remain notebook components. Root operational orchestration owns any later reviewed Gym submission.

Explicit run variables: `PA3_RUN_FINAL_V011=1`, optional `PA3_RESUME_FINAL_V011=1`, and mandatory `PA3_FINAL_V011_SELECTION_SHA256`, `PA3_FINAL_V011_D_RESULT_SHA256`, `PA3_FINAL_V011_SOURCE_SHA256`. Selection hashing is compact sorted canonical JSON (`separators=(',',':'), allow_nan=False`); the D-result digest is raw file bytes; notebook code hashing joins executable cell sources with one newline. Main API: `v011_run_final(project_root='.', *, resume=False, expected_selection_sha256=..., expected_confirmation_file_sha256=..., expected_source_code_sha256=...)`. `v011_final_selfcheck()` uses only synthetic vectors and temporary files and does not launch training.


In [ ]:
# Final V011 definitions. Default execution reads no assignment data or outcomes.
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import copy, fcntl, gc, hashlib, json, math, platform, re, time, uuid
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch

V011_FINAL_VERSION = 'v011-conditional-all-fit-inference-v1'
V011_FINAL_NOTEBOOK = 'final_fit_v011.ipynb'
V011_FINAL_OUT = 'outputs/confirmation_v011/final_all'
V011_LOCK_FILE = 'outputs/confirmation_v011/locked_before_D.json'
V011_D_RESULT_FILE = 'outputs/confirmation_v011/confirmation_result.json'
V011_D_SPLIT_FILE = 'outputs/confirmation_v011/D_split.json'
V011_TRAIN_SHA = '2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7'
V011_TEST_SHA = '1ed86b32480e93397cc961dfd09af2d0bedfbab901921eac44de8ed78c3f1672'
V011_SAMPLE_SHA = 'de4c137a22804654b597723f32470b24f23d9ef662205eca0fe31b68970f0594'
V011_PRIOR_DIR = 'outputs/improvements_v005/confirmation_round1/final_all'
V011_PRIOR_MANIFEST_SHA = 'a9e9e48881c169587728519c5a8ca400ab5a13b6c760fb635457308ecf2615fd'
V011_PRIOR_CSV_SHA = 'ff149f3ab87e1e52b948f536c9fdf9a526471ac0723b6b785bf84c053ebed607'
V011_ORIGINALS = {
 'original_compact': dict(name='nn__capacity_bilstm128', epochs=193,
  checkpoint_sha256='19897b624599c57f2e4a7f976b2d2fdd4d0b5b1ba3f3dbf6884198ebdef23883',
  record_sha256='b11f9f3b566a6ab48f2fe6c21361d9cef37ae4af9fa48ba6c2a70f7ad048b922',
  report_sha256='b5832eeda701776e7fd3a85cbdd37ebc00887493cae6833d10220741a50c9869'),
 'deep': dict(name='nn__capacity_deep_bilstm', epochs=189,
  checkpoint_sha256='591e3a778af84bfe7fa0f6592f8595fa2c969bdbb37fc0a36a2a251eacb4aad0',
  record_sha256='9cfc78a3dcc989f0af57e5d1be55fba5e74c96ba2e6981766ecd0a8e59b3060c',
  report_sha256='b1282b4fe8351c9871a7a86857fbe48992e36e1a3582e60c055078b3c0b4b181')}
V011_FINAL_COMPONENTS = {'feature_components_v005.ipynb': 'eb19eed69eb54cd5d087a9a678cf98701fbe9bc1a3cf85016270754c014593a6', 'sequence_components_v005.ipynb': 'e6698a7f6e931af8a066c227058203b7e29ada68624833b700293532a30bf1b2', 'attention_components_v008.ipynb': '47c43f75bf58385771de89e37b0fed05054f8a6d6e5bd1ac3e15f4891111625b', 'memory_components_v008.ipynb': 'b7dd6c6434ada27ae5cee3e098aa113614ac891e0ecd4e7e25917ab92ad47d8a', 'offset_components_v009.ipynb': '1000c5311fcf4cfc348fd5b99975fcd5816650d1616ace31eace5407ccd2a1b3', 'tcn_components_v009.ipynb': 'd4e08e56a78a8d1d8e45ee0843302d7d2a1c9b35ccc1ed8628ae4f1239da486e', 'tcn_fast_components_v009.ipynb': '51d7438d2bd719e5609ed8530a4ad9cf9009943498641bf9488c9e81b1a5024d', 'training_components_v008.ipynb': 'a65618d1886a7ad3d1753f1f752b449114a57dcb75084f4f6cd11fc084228bda'}
V011_SEED = 20261102
V011_KNOWN_COLUMNS = ['id','breath_id','R','C','time_step','u_in','u_out']
V011_DTYPES = dict(id='int64',breath_id='int32',R='int8',C='int8',time_step='float32',u_in='float32',u_out='int8',pressure='float32')


def v011_require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def v011_file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1048576), b''):
            digest.update(block)
    return digest.hexdigest()


def v011_json_sha(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()


def v011_legacy_json_sha(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()


def v011_array_sha(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()


def v011_code_sha(path):
    book = json.loads(Path(path).read_text())
    return hashlib.sha256('\n'.join(''.join(cell['source']) for cell in book['cells']
        if cell['cell_type']=='code').encode()).hexdigest()


def v011_local(root, value):
    root = Path(root).resolve()
    text = str(value)
    marker = '/ventilator-pressure-prediction/'
    if marker in text:
        text = text.split(marker,1)[1]
    path = Path(text)
    path = path if path.is_absolute() else root/path
    path = path.resolve()
    v011_require(path.is_relative_to(root), 'Artifact escapes project root: '+str(value))
    return path


def v011_hex_digest(value, name):
    v011_require(isinstance(value,str) and re.fullmatch('[0-9a-f]{64}',value) is not None,
        name+' must be an externally frozen lowercase SHA256')
    return value


def v011_verify_file(root, path, expected):
    path = v011_local(root,path)
    v011_require(v011_file_sha(path)==v011_hex_digest(expected,str(path)), 'File hash changed: '+str(path))
    return path


def v011_write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    try:
        temporary.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')
        os.replace(temporary,path)
    finally:
        if temporary.exists(): temporary.unlink()


def v011_save_array(path, value):
    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    try:
        with temporary.open('wb') as stream: np.save(stream,value,allow_pickle=False)
        os.replace(temporary,path)
    finally:
        if temporary.exists(): temporary.unlink()


def v011_immutable_json(path, value, resume):
    path = Path(path)
    if path.exists():
        v011_require(resume and json.loads(path.read_text())==value,'Frozen final design changed or resume omitted')
    else:
        v011_write_json(path,value)


def v011_event(directory, name, **fields):
    with (Path(directory)/'events.jsonl').open('a') as stream:
        stream.write(json.dumps(dict(time_utc=datetime.now(timezone.utc).isoformat(),event=name,**fields),allow_nan=False)+'\n')




In [ ]:
# Selection and D evidence are consumed without reading D labels or searching weights.
def v011_validate_roster(members, winner=True):
    v011_require(isinstance(members,list) and members,'Missing locked members')
    ids = [member['member_id'] for member in members]
    v011_require(len(ids)==len(set(ids)), 'Duplicate locked member')
    weights = {member['member_id']:float(member['weight']) for member in members}
    v011_require(all(math.isfinite(w) and w>0 for w in weights.values())
        and math.isclose(sum(weights.values()),1.,rel_tol=0.,abs_tol=1e-15),'Invalid frozen weights')
    choices = [dict(deep=.5,original_compact=.5)] if not winner else [
        dict(deep=.5,offset=.5),dict(deep=.5,tcn=.5),
        dict(deep=.5,original_compact=1/6,curve=1/6,memory=1/6),
        dict(deep=.5,original_compact=1/8,curve=1/8,memory=1/8,attention=1/8),
        dict(deep=.5,original_compact=1/8,curve=1/8,memory=1/8,offset=1/8)]
    v011_require(any(weights.keys()==choice.keys() and all(math.isclose(weights[k],choice[k],rel_tol=0.,abs_tol=1e-15)
        for k in choice) for choice in choices),'Roster/weights differ from the approved fixed candidate packages')
    for member in members:
        ident = member['member_id']; recipe = member['recipe']
        v011_require(member['factory_id']==ident, 'Unknown factory mapping')
        v011_require(member['kind']==('v5' if ident in V011_ORIGINALS else 'v8'),'Wrong checkpoint kind')
        v011_require(recipe['seed']==V011_SEED and recipe['feature_mode']=='combined','Frozen seed/features changed')
        cfg = recipe['train_cfg']; rates = recipe['lr_schedule']; count = recipe['selected_epochs']
        v011_require(type(count) is int and 0<count<=200 and len(rates)==count,'Invalid frozen epoch/rate prefix')
        v011_require(cfg['max_epochs']==200 and cfg['batch_size']==256 and cfg['optimizer']=='Adam'
            and cfg['weight_decay']==0. and cfg['loss']=='l1' and cfg['scaler_phase']=='all'
            and not cfg['residual_baseline'] and cfg['scheduler'] in ('replay','fixed')
            and cfg['patience']==0 and len(cfg['frozen_lr_schedule'])==200
            and rates==cfg['frozen_lr_schedule'][:count], 'Frozen original training controls changed')
        v011_require(all(math.isfinite(float(rate)) and rate>=0 for rate in rates),'Nonfinite frozen rate')
        v011_require(recipe['loss_kind']==('curve_l1' if ident=='curve' else 'l1'),'Unexpected combined/changed objective')
        v011_require(recipe['model_spec']['dropout']==.15,'Original dropout changed')
        if ident in V011_ORIGINALS:
            v011_require(count==V011_ORIGINALS[ident]['epochs'],'Accepted original epoch prefix changed')
    return weights


def v011_blend(predictions, members):
    first = np.asarray(predictions[members[0]['member_id']],dtype=np.float64)
    result = np.zeros(first.shape,dtype=np.float64)
    for member in members:
        values = np.asarray(predictions[member['member_id']],dtype=np.float64)
        v011_require(values.shape==first.shape and np.isfinite(values).all(),'Member predictions misaligned/nonfinite')
        result += float(member['weight'])*values
    v011_require(np.isfinite(result).all(),'Nonfinite weighted native prediction')
    return result


def v011_gate_decision(lock, result, expected_selection_sha):
    selection_sha = v011_json_sha(lock)
    v011_require(selection_sha==expected_selection_sha==result['selection_sha256'],'Locked A choice changed')
    v011_require(lock['schema']=='confirmation_v011_locked_before_D','Unknown frozen lock schema')
    v011_require(lock['approved_train_sha256']==result['approved_train_sha256']==V011_TRAIN_SHA,'Approved train hash changed')
    v011_require(result['status']=='complete' and result['all_required_members_completed']
        and result['original_A_validation_excluded'] and result['frozen_A_choice']
        and result['no_runner_up_after_failure'] and result['confirmation_is_resampling_evidence'],
        'D is incomplete or required confirmation constraints failed')
    v011_require(result['winner_name']==lock['winner']['name'],'D winner differs from the frozen lowest-A choice')
    v011_validate_roster(lock['winner']['members']); v011_validate_roster(lock['baseline']['members'],winner=False)
    v011_require(lock['original_A_validation_excluded'] and lock['D_is_resampling_confirmation']
        and lock['pressure_labels_parsed_before_lock'] is False and lock['D_outcomes_read_before_lock'] is False
        and result['no_D_weight_or_epoch_selection'] is True and result['schema']=='confirmation_v011_result'
        and result['main_code_sha256']==lock['main_code_sha256'],'Frozen confirmation protocol changed')
    rule=lock['confirmation_rule']
    v011_require(rule['strictly_positive_gain'] and rule['whole_breath_bootstrap_upper95_below_zero']
        and rule['bootstrap_seed']==20261201 and rule['bootstrap_draws']==1000
        and rule['no_runner_up_after_failure'] and rule['no_weight_or_epoch_search'],'Confirmation rule changed')
    candidates=lock['completed_A_candidates']
    expected_names={'shared_breath_offset','full_context_tcn','deep_half_compact_curve_memory',
        'deep_half_compact_curve_memory_attention','deep_half_compact_curve_memory_offset'}
    v011_require(len(candidates)==5 and {row['name'] for row in candidates}==expected_names,
        'All five predeclared A candidates must finish before selection')
    qualifying=[]
    for row in candidates:
        eligible=float(row['A_gain'])>=.003 and float(row['A_interval']['bootstrap_95_high'])<0.
        v011_require(math.isclose(.19656260982217663-float(row['A_MAE']),float(row['A_gain']),rel_tol=0.,abs_tol=1e-12)
            and row['eligible']==eligible,'Recorded A qualification changed')
        if eligible: qualifying.append(row)
    v011_require(qualifying and min(qualifying,key=lambda row:(row['A_MAE'],row['name']))['name']==lock['winner']['name'],
        'Final choice is not the frozen single lowest-A qualifying candidate')
    winner = lock['winner']
    v011_require(float(winner['A_gain'])>=.003 and float(winner['A_interval']['bootstrap_95_high'])<0.,'A selection did not qualify')
    decision = result['decision']; interval = decision['interval']
    numbers = [decision['gain'],decision['baseline_MAE'],decision['candidate_MAE'],interval['difference'],
        interval['bootstrap_95_low'],interval['bootstrap_95_high']]
    v011_require(all(math.isfinite(float(x)) for x in numbers),'Nonfinite confirmation evidence')
    gain = float(decision['baseline_MAE'])-float(decision['candidate_MAE'])
    v011_require(decision['passed'] is True and gain>0. and float(interval['bootstrap_95_high'])<0.
        and float(interval['bootstrap_95_low'])<=float(interval['bootstrap_95_high'])
        and math.isclose(gain,float(decision['gain']),rel_tol=0.,abs_tol=1e-12)
        and math.isclose(float(interval['difference']),-gain,rel_tol=0.,abs_tol=1e-12),
        'Frozen D confirmation did not pass: no final fit is allowed')
    v011_require(interval['seed']==20261201 and interval['repeats']==1000
        and interval['point_count_weighted'] is True,'Confirmation bootstrap specification changed')
    v011_require(result['source_fingerprints']==lock['source_fingerprints'],'Confirmation component sources changed')
    return selection_sha


def v011_load_gate(root, expected_selection_sha, expected_result_sha, expected_source_sha):
    root = Path(root).resolve()
    for value,name in [(expected_selection_sha,'selection'),(expected_result_sha,'D result'),(expected_source_sha,'final source')]:
        v011_hex_digest(value,name)
    v011_require(v011_code_sha(root/V011_FINAL_NOTEBOOK)==expected_source_sha,'Final notebook code changed')
    lock_path = root/V011_LOCK_FILE
    result_path = v011_verify_file(root,V011_D_RESULT_FILE,expected_result_sha)
    lock, result = json.loads(lock_path.read_text()),json.loads(result_path.read_text())
    selection_sha = v011_gate_decision(lock,result,expected_selection_sha)
    v011_require(v011_file_sha(lock_path)==result['locked_before_D_file_sha256'],'Raw locked selection file changed')
    v011_require(v011_code_sha(root/'confirmation_v011.ipynb')==lock['main_code_sha256'],'Confirmation implementation changed')
    for filename,digest in lock['source_fingerprints'].items():
        v011_require('/' not in filename and filename.endswith('.ipynb'),'Unexpected component filename')
        v011_require(v011_code_sha(root/'research'/filename)==v011_hex_digest(digest,filename),'Locked component changed: '+filename)
    for filename,digest in lock['selection_input_files'].items():
        v011_verify_file(root,filename,digest)
    split_path = v011_verify_file(root,V011_D_SPLIT_FILE,lock['D_split']['split_file_sha256'])
    split = json.loads(split_path.read_text())
    v011_require(result['D_split']['split_file_sha256']==v011_file_sha(split_path)
        and result['D_split']['group_hashes']==lock['D_split']['group_hashes']==split['group_hashes']
        and result['D_split']['counts']==lock['D_split']['counts']==split['counts']
        and result['D_split']==lock['D_split'] and split['method']==lock['D_split']['method']
        and split['index_hashes']==lock['D_split']['index_hashes']
        and split['seed']==lock['D_split']['seed']==20261202,'D split identity changed')
    required = {m['member_id']:m for m in lock['required_members']}
    expected_ids = {m['member_id'] for m in lock['winner']['members']+lock['baseline']['members']}
    v011_require(set(required)==expected_ids==set(result['member_artifacts']),'Required confirmation roster changed')
    for member in lock['winner']['members']+lock['baseline']['members']:
        canonical = dict(member); canonical.pop('weight',None)
        recorded = dict(required[member['member_id']]); recorded.pop('weight',None)
        v011_require(canonical==recorded,'Member recipe differs across locked rosters')
        v011_require(member['source_fingerprints']==lock['source_fingerprints'],'Member source fingerprints differ')
    row_spec = result['ensemble_artifacts']['row_ids']
    row_path = v011_verify_file(root,row_spec['file'],row_spec['file_sha256'])
    row_ids = np.load(row_path,allow_pickle=False)
    v011_require(row_ids.ndim==1 and row_ids.dtype==np.int64 and len(row_ids)>0
        and len(np.unique(row_ids))==len(row_ids) and v011_array_sha(row_ids)==row_spec['array_sha256'],
        'D row-ID artifact invalid')
    predictions = {}
    for ident,artifact in result['member_artifacts'].items():
        member = required[ident]
        v011_verify_file(root,artifact['checkpoint_path'],artifact['checkpoint_sha256'])
        report_path = v011_verify_file(root,artifact['report_path'],artifact['report_sha256'])
        report = json.loads(report_path.read_text())
        v011_require(artifact['training_breaths']==54323 and artifact['selected_epochs']==member['recipe']['selected_epochs']
            and report['training_breaths']==54323 and report['completed'] and not report['validation_used']
            and report['executed_epochs']==member['recipe']['selected_epochs']
            and report['selected_lr_schedule']==member['recipe']['lr_schedule'], 'Incomplete or altered cold D fit')
        path = v011_verify_file(root,artifact['prediction_file'],artifact['prediction_file_sha256'])
        prediction = np.load(path,allow_pickle=False)
        v011_require(prediction.shape==row_ids.shape and prediction.dtype==np.float64 and np.isfinite(prediction).all()
            and v011_array_sha(prediction)==artifact['prediction_array_sha256']
            and artifact['row_ids_array_sha256']==row_spec['array_sha256'],'D prediction alignment/hash changed')
        predictions[ident] = prediction
    for role in ['baseline','candidate']:
        artifact = result['ensemble_artifacts'][role]
        path = v011_verify_file(root,artifact['prediction_file'],artifact['file_sha256'])
        saved = np.load(path,allow_pickle=False)
        roster = lock['baseline']['members'] if role=='baseline' else lock['winner']['members']
        blend = v011_blend(predictions,roster)
        v011_require(saved.dtype==np.float64 and saved.shape==row_ids.shape and np.isfinite(saved).all()
            and v011_array_sha(saved)==artifact['array_sha256'] and np.allclose(saved,blend,rtol=0.,atol=1e-12),
            'D ensemble no longer matches frozen weights: '+role)
    return lock,result,split,dict(selection_sha256=selection_sha,locked_before_D_file_sha256=v011_file_sha(lock_path),
        confirmation_result_sha256=expected_result_sha,final_source_code_sha256=expected_source_sha,
        D_split_file_sha256=v011_file_sha(split_path),D_row_ids_array_sha256=row_spec['array_sha256'])




In [ ]:
# Model definitions remain in the existing notebook components.
def v011_load_components(root, locked_sources):
    root = Path(root)
    for filename,expected in V011_FINAL_COMPONENTS.items():
        v011_require(locked_sources.get(filename)==expected and v011_code_sha(root/'research'/filename)==expected,
            'Final component/source freeze differs: '+filename)
    for filename in ['feature_components_v005.ipynb','sequence_components_v005.ipynb',
        'attention_components_v008.ipynb','memory_components_v008.ipynb','offset_components_v009.ipynb',
        'tcn_components_v009.ipynb','tcn_fast_components_v009.ipynb','training_components_v008.ipynb']:
        book = json.loads((root/'research'/filename).read_text())
        for index,cell in enumerate(book['cells']):
            if cell['cell_type']=='code': exec(compile(''.join(cell['source']),filename+':'+str(index),'exec'),globals())


def _v011_curve_factory(spec,n_features,prep): return V5PressureRNN(spec,n_features)
def _v011_memory_factory(spec,n_features,prep): return v8_memory_model(spec,n_features,prep)
def _v011_attention_factory(spec,n_features,prep): return v8_attention_model(spec,n_features)
def _v011_offset_factory(spec,n_features,prep): return v9_offset_model(spec,n_features,prep)
def _v011_tcn_factory(spec,n_features,prep): return v9_fast_tcn_model(spec,n_features,prep)
for _v011_factory in [_v011_curve_factory,_v011_memory_factory,_v011_attention_factory,_v011_offset_factory,_v011_tcn_factory]:
    _v011_factory.__module__ = 'final_fit_v011'


def v011_factory(ident):
    factories = dict(curve=_v011_curve_factory,memory=_v011_memory_factory,
        attention=_v011_attention_factory,offset=_v011_offset_factory,tcn=_v011_tcn_factory)
    v011_require(ident in factories,'Only selected new V8-format members may be fitted')
    return factories[ident]


def v011_check_fixed_payload(payload, member, data_hashes, expected_prep, n_breaths=67905, scored_points=2061922):
    recipe = member['recipe']; rates = recipe['lr_schedule']; n = recipe['selected_epochs']
    v011_require(payload['completed'] and payload['epoch']==n and len(payload['epochs'])==n
        and payload['seed']==V011_SEED and payload['model_spec']==canonical_model_spec(recipe['model_spec'])
        and payload['config']==_train_configuration(recipe['train_cfg'])
        and payload['selected_lr_schedule']==rates and payload['termination']=='fixed_A_budget'
        and payload['optimizer_steps']==math.ceil(n_breaths/recipe['train_cfg']['batch_size'])*n,
        'Final checkpoint budget/recipe incomplete or changed')
    fingerprint = payload['fingerprint']
    for key,value in data_hashes.items():
        v011_require(fingerprint[key]==value,'Final checkpoint all-training data fingerprint changed: '+key)
    v011_require(fingerprint['mode']=='fixed' and fingerprint['model_spec']==canonical_model_spec(recipe['model_spec'])
        and fingerprint['train_cfg']==_train_configuration(recipe['train_cfg']) and fingerprint['seed']==V011_SEED
        and fingerprint['feature_names']==expected_prep['feature_names']
        and fingerprint['extra']==dict(lr_schedule=rates),'Final checkpoint training fingerprint changed')
    prep = _validated_preprocessor(payload['preprocess'])
    v011_require(prep['fitted_training_breaths']==n_breaths and prep['fitted_inspiratory_points']==scored_points
        and _json_safe(prep)==_json_safe(expected_prep),'Final preprocessing is not the verified all-training-only fit')
    v011_require([record['lr_used'] for record in payload['epochs']]==rates,'Checkpoint executed rate list changed')
    return prep



def v011_verify_legacy_lock(root, prior_manifest):
    """V005 stores a spaced-canonical digest excluding its own lock_sha256 field."""
    path=v011_local(root,prior_manifest['lock_file'])
    legacy=json.loads(path.read_text())
    expected=v011_hex_digest(prior_manifest['lock_sha256'],'legacy lock canonical digest')
    unsigned={key:value for key,value in legacy.items() if key!='lock_sha256'}
    v011_require(legacy.get('lock_sha256')==expected
        and v011_legacy_json_sha(unsigned)==expected
        and legacy.get('design_sha256')==prior_manifest['design_sha256'],
        'Accepted original canonical lock provenance changed')
    return dict(file=str(path),file_sha256=v011_file_sha(path),canonical_sha256=expected,
        convention='V005 json.dumps(sort_keys=True), excluding lock_sha256')

def v011_reuse_original(root, member, all_data, prior_manifest):
    ident = member['member_id']; frozen = V011_ORIGINALS[ident]; name = frozen['name']
    base = Path(root)/V011_PRIOR_DIR
    record_path = v011_verify_file(root,base/('record_'+name+'.json'),frozen['record_sha256'])
    report_path = v011_verify_file(root,base/('report_'+name+'.json'),frozen['report_sha256'])
    record,report = json.loads(record_path.read_text()),json.loads(report_path.read_text())
    checkpoint = v011_verify_file(root,base/(name+'.pt'),frozen['checkpoint_sha256'])
    v011_require(prior_manifest['model_hashes'][name]==frozen['checkpoint_sha256']
        and v011_local(root,prior_manifest['model_files'][name])==checkpoint
        and v011_local(root,prior_manifest['model_records'][name])==record_path
        and prior_manifest['recipes'][name]==record['recipe']==record['identity']['recipe']
        and record['recipe']['model_spec']==member['recipe']['model_spec']
        and record['recipe']['train_cfg']==member['recipe']['train_cfg']
        and record['recipe']['seed']==member['recipe']['seed']
        and record['recipe']['feature_mode']==member['recipe']['feature_mode'],'Accepted original recipe no longer matches frozen A member')
    v011_require(record['status']=='complete' and record['stage']=='all' and record['kind']=='v5_nn'
        and record['training_breaths']==67905 and record['scored_training_rows']==2061922
        and not record['validation_feedback'] and not record['has_independent_holdout']
        and record['fit_group_sha256']==all_data['group_sha256']==record['identity']['fit_group_sha256']
        and record['identity_sha256']==v011_legacy_json_sha(record['identity'])
        and record['identity']['selected_lr_schedule']==record['selected_lr_schedule']==member['recipe']['lr_schedule']
        and record['checkpoint_sha256']==frozen['checkpoint_sha256'] and record['report_sha256']==frozen['report_sha256'],
        'Accepted original provenance/count/schedule changed')
    v011_require(report['completed'] and not report['validation_used'] and report['training_breaths']==67905
        and report['executed_epochs']==frozen['epochs'] and report['completed_epochs']==frozen['epochs']
        and report['selected_lr_schedule']==report['executed_lr_schedule']==member['recipe']['lr_schedule']
        and report['model_spec']==member['recipe']['model_spec'] and report['train_cfg']==member['recipe']['train_cfg']
        and report['preprocessing']==_json_safe(all_data['prep']),'Accepted original report/preprocessing changed')
    payload = torch.load(checkpoint,map_location='cpu',weights_only=False)
    v011_require(payload.get('format_version')==5,'Accepted original checkpoint format changed')
    v011_check_fixed_payload(payload,member,all_data['data_hashes'],all_data['prep'])
    model = V5PressureRNN(payload['model_spec'],47); model.load_state_dict(payload['state_dict'],strict=True); del model,payload
    return dict(member_id=ident,kind='v5',reused_accepted_all_training_checkpoint=True,
        checkpoint_path=str(checkpoint),checkpoint_sha256=frozen['checkpoint_sha256'],
        recipe=copy.deepcopy(member['recipe']),training_breaths=67905,scored_training_points=2061922,
        group_sha256=all_data['group_sha256'],preprocessing_sha256=_v8_statistics_sha256(all_data['prep']),
        prior_record_file=str(record_path),prior_record_sha256=frozen['record_sha256'],
        prior_report_file=str(report_path),prior_report_sha256=frozen['report_sha256'],elapsed_seconds=0.)


def v011_load_all_training(root, lock, split):
    root = Path(root)
    for name,digest in [('train.csv',V011_TRAIN_SHA),('test.csv',V011_TEST_SHA),('sample_submission.csv',V011_SAMPLE_SHA)]:
        v011_verify_file(root,root/'data'/name,digest)
    frame = pd.read_csv(root/'data/train.csv',dtype=V011_DTYPES)
    v011_require(len(frame)==67905*80,'Approved train row count changed')
    breath_ids = _validate_feature_frame_v005(frame)
    v011_require(len(breath_ids)==67905,'Approved training group count changed')
    y = frame.pressure.to_numpy(copy=True).reshape(-1,80)
    out = frame.u_out.to_numpy(copy=True).reshape(-1,80)
    row_ids = frame.id.to_numpy(copy=True)
    v011_require(np.isfinite(y).all() and int((out==0).sum())==2061922,'Approved target/count changed')
    groups = {name:np.asarray(values,dtype=np.int64) for name,values in split['indices'].items()}
    for name in ['A_fit','A_validation','D_fit','D_validation']:
        ids = np.asarray(split['breath_ids'][name],dtype=np.int32); indices=groups[name]
        v011_require(len(indices)==split['counts'][name] and np.array_equal(indices,np.sort(np.unique(indices)))
            and (indices>=0).all() and (indices<67905).all() and np.array_equal(breath_ids[indices],ids)
            and v011_array_sha(ids)==split['group_hashes'][name]
            and v011_array_sha(indices)==split['index_hashes'][name],'Frozen group/index identity changed: '+name)
    v011_require(len(groups['A_fit'])==61114 and len(groups['A_validation'])==6791
        and len(groups['D_fit'])==54323 and len(groups['D_validation'])==6791
        and not np.intersect1d(groups['D_fit'],groups['D_validation']).size
        and np.array_equal(np.sort(np.r_[groups['D_fit'],groups['D_validation']]),groups['A_fit'])
        and not np.intersect1d(groups['A_fit'],groups['A_validation']).size
        and np.array_equal(np.sort(np.r_[groups['A_fit'],groups['A_validation']]),np.arange(67905)),
        'D containment/original A exclusion changed')
    with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz',allow_pickle=False) as original:
        for name in ['A_fit','A_validation']:
            v011_require(np.array_equal(np.sort(original[name]),np.sort(breath_ids[groups[name]])), 'Original A split changed')
    # Independently reconstruct the predeclared RC-stratified D split using known
    # controls only; labels are used solely by the subsequent all-training fitter.
    from sklearn.model_selection import train_test_split
    rc = frame[['R','C']].to_numpy()[::80]
    joint = np.searchsorted([5,20,50],rc[:,0])*3+np.searchsorted([10,20,50],rc[:,1])
    dfit,dval = train_test_split(groups['A_fit'],test_size=6791,random_state=20261202,stratify=joint[groups['A_fit']])
    v011_require(np.array_equal(np.sort(dfit),groups['D_fit']) and np.array_equal(np.sort(dval),groups['D_validation']),
        'Fresh grouped confirmation seed/stratification changed')
    known = frame[V011_KNOWN_COLUMNS].copy(); del frame
    x,names,xids = make_sequence_features(known,'combined'); del known; gc.collect()
    v011_require(x.shape==(67905,80,47) and names==feature_names_v005('combined') and np.array_equal(xids,breath_ids),
        'All-training known feature schema/order changed')
    recipe = lock['winner']['members'][0]['recipe']
    prep = fit_preprocessor(x,y,out,recipe['model_spec'],names,scaler_phase='all',residual_baseline=False)
    return dict(x=x,y=y,out=out,names=names,breath_ids=breath_ids,row_ids=row_ids,prep=prep,
        group_sha256=v011_array_sha(breath_ids),row_ids_sha256=v011_array_sha(row_ids),
        data_hashes=dict(x_sha256=_array_sha256(x),y_sha256=_array_sha256(y),out_sha256=_array_sha256(out)))


def v011_fit_new_member(root, directory, member, data, gate, source_fingerprints, resume):
    ident=member['member_id']; recipe=member['recipe']; stage=Path(directory)/ident; stage.mkdir(exist_ok=True)
    checkpoint=stage/'model.pt'; record_path=stage/'record.json'; report_path=stage/'report.json'
    arm='v011_final_'+ident; factory=v011_factory(member['factory_id'])
    identity=dict(version=V011_FINAL_VERSION,gate=gate,member=member,all_data_hashes=data['data_hashes'],
        training_group_sha256=data['group_sha256'],preprocessing_sha256=_v8_statistics_sha256(data['prep']),
        source_fingerprints=source_fingerprints,arm=arm)
    if record_path.exists():
        v011_require(resume,'Completed final member exists; explicit resume required')
        record=json.loads(record_path.read_text())
        v011_require(record['identity']==identity and record['identity_sha256']==v011_json_sha(identity),'Final member identity changed')
        v011_verify_file(root,record['checkpoint_path'],record['checkpoint_sha256'])
        v011_verify_file(root,record['report_file'],record['report_sha256'])
    else:
        began=time.monotonic(); last=_last_path(checkpoint)
        v011_require(not last.exists() or resume,'Partial final checkpoint requires explicit resume')
        v011_event(directory,'new_member_final_fit_started',member_id=ident,selected_epochs=recipe['selected_epochs'],
            identity_sha256=v011_json_sha(identity),training_breaths=67905,validation_used=False)
        report=fit_fixed_v008(data['x'],data['y'],data['out'],recipe['model_spec'],recipe['train_cfg'],
            recipe['seed'],recipe['lr_schedule'],checkpoint,epoch_log_path=stage/'epochs.jsonl',
            feature_names=data['names'],device='cuda',resume=resume and last.exists(),model_factory=factory,
            loss_kind=recipe['loss_kind'],arm=arm,source_fingerprints=source_fingerprints)
        v011_require(report['completed'] and report['executed_epochs']==recipe['selected_epochs']
            and report['training_breaths']==67905 and not report['validation_used'],'Final fit incomplete; exact state retained')
        v011_write_json(report_path,report)
        record=dict(member_id=ident,kind='v8',identity=identity,identity_sha256=v011_json_sha(identity),
            reused_accepted_all_training_checkpoint=False,recipe=copy.deepcopy(recipe),training_breaths=67905,
            scored_training_points=2061922,group_sha256=data['group_sha256'],
            preprocessing_sha256=_v8_statistics_sha256(data['prep']),checkpoint_path=str(checkpoint),
            checkpoint_sha256=v011_file_sha(checkpoint),report_file=str(report_path),report_sha256=v011_file_sha(report_path),
            elapsed_seconds=time.monotonic()-began,arm=arm,source_fingerprints=source_fingerprints)
        v011_write_json(record_path,record)
    report=json.loads(report_path.read_text())
    v011_require(report['completed'] and report['executed_epochs']==recipe['selected_epochs']
        and report['selected_lr_schedule']==report['executed_lr_schedule']==recipe['lr_schedule']
        and not report['validation_used'] and report['training_breaths']==67905,'Saved final report is incomplete or changed')
    model,prep,payload=load_checkpoint_v008(checkpoint,device='cpu',model_factory=factory,
        expected_arm=arm,expected_loss_kind=recipe['loss_kind'],source_fingerprints=source_fingerprints)
    v011_check_fixed_payload(payload,member,data['data_hashes'],data['prep'])
    del model,prep,payload; gc.collect(); torch.cuda.empty_cache()
    return record


def v011_infer_members(root, directory, members, records, source_fingerprints, resume):
    root=Path(root); directory=Path(directory)
    test=pd.read_csv(root/'data/test.csv',usecols=V011_KNOWN_COLUMNS,dtype={k:V011_DTYPES[k] for k in V011_KNOWN_COLUMNS})
    sample=pd.read_csv(root/'data/sample_submission.csv',dtype=dict(id='int64',pressure='float64'))
    v011_require(list(sample.columns)==['id','pressure'] and len(test)==len(sample)==603600
        and test.id.is_unique and sample.id.is_unique and np.array_equal(test.id.to_numpy(),sample.id.to_numpy()),
        'Approved test/sample603600 IDs are not exactly aligned')
    x,names,groups=make_sequence_features(test,'combined')
    v011_require(x.shape==(7545,80,47) and names==feature_names_v005('combined'),'Test known feature schema changed')
    ids=test.id.to_numpy(copy=True); del test,sample; gc.collect()
    predictions={}; prediction_records={}
    for member in members:
        ident=member['member_id']; record=records[ident]
        v011_verify_file(root,record['checkpoint_path'],record['checkpoint_sha256'])
        began=time.monotonic(); path=directory/(ident+'_test_predictions.npy')
        meta_path=directory/(ident+'_test_predictions.json')
        meta=json.loads(meta_path.read_text()) if meta_path.exists() else None
        if path.exists() or meta is not None:
            v011_require(resume,'Final member inference artifacts exist; explicit resume required')
        if meta is not None:
            v011_require(meta['checkpoint_sha256']==record['checkpoint_sha256'] and meta['test_sha256']==V011_TEST_SHA
                and meta['row_ids_sha256']==v011_array_sha(ids),'Saved inference identity changed')
        if path.exists() and meta is not None:
            v011_verify_file(root,path,meta['file_sha256']); prediction=np.load(path,allow_pickle=False)
            v011_require(v011_array_sha(prediction)==meta['array_sha256'],'Saved prediction array changed')
        else:
            # Recovery after interruption between publishing the array and its
            # metadata recomputes this fixed checkpoint; it never trusts an orphan.
            if record['kind']=='v5':
                prediction=predict_checkpoint(record['checkpoint_path'],x,device='cuda',batch_size=member['recipe']['train_cfg']['prediction_batch_size'])
            else:
                prediction=predict_checkpoint_v008(record['checkpoint_path'],x,device='cuda',
                    batch_size=member['recipe']['train_cfg']['prediction_batch_size'],model_factory=v011_factory(member['factory_id']),
                    expected_arm=record['arm'],expected_loss_kind=member['recipe']['loss_kind'],source_fingerprints=source_fingerprints)
            prediction=np.asarray(prediction,dtype=np.float64).reshape(-1)
            v011_require(prediction.shape==ids.shape and np.isfinite(prediction).all(),'Final native prediction invalid')
            if path.exists():
                v011_require(np.array_equal(np.load(path,allow_pickle=False),prediction),
                    'Incomplete cached inference differs from frozen checkpoint; artifact preserved for review')
            else:
                if meta is not None:
                    v011_require(v011_array_sha(prediction)==meta['array_sha256'],'Recovered inference no longer matches saved metadata')
                v011_save_array(path,prediction)
            if meta is None:
                meta=dict(member_id=ident,checkpoint_sha256=record['checkpoint_sha256'],test_sha256=V011_TEST_SHA,
                    row_ids_sha256=v011_array_sha(ids),file=str(path),file_sha256=v011_file_sha(path),
                    array_sha256=v011_array_sha(prediction),dtype=prediction.dtype.str,shape=list(prediction.shape),
                    elapsed_seconds=time.monotonic()-began)
                v011_write_json(meta_path,meta)
            else:
                v011_verify_file(root,path,meta['file_sha256'])
        v011_require(prediction.dtype==np.float64 and prediction.shape==ids.shape and np.isfinite(prediction).all(),'Saved native prediction invalid')
        predictions[ident]=prediction; prediction_records[ident]=meta
        gc.collect(); torch.cuda.empty_cache()
    del x; gc.collect()
    return ids,groups,predictions,prediction_records


def v011_write_submission(directory, ids, prediction, resume=False):
    directory=Path(directory); path=directory/'submission.csv'
    ids=np.asarray(ids,dtype=np.int64); prediction=np.asarray(prediction,dtype=np.float64)
    v011_require(ids.ndim==1 and prediction.shape==ids.shape and np.isfinite(prediction).all()
        and len(np.unique(ids))==len(ids),'Submission row/prediction contract failed')
    if path.exists():
        v011_require(resume,'New final CSV already exists; explicit resume required')
    else:
        temporary=path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
        try:
            pd.DataFrame(dict(id=ids,pressure=prediction)).to_csv(temporary,index=False,float_format='%.17g')
            os.replace(temporary,path)
        finally:
            if temporary.exists(): temporary.unlink()
    loaded=pd.read_csv(path,dtype=dict(id='int64',pressure='float64'),float_precision='round_trip')
    v011_require(list(loaded.columns)==['id','pressure'] and np.array_equal(loaded.id.to_numpy(),ids)
        and np.array_equal(loaded.pressure.to_numpy(),prediction),'CSV values/IDs do not round-trip exactly')
    return dict(file=str(path),file_sha256=v011_file_sha(path),rows=len(ids),
        row_ids_array_sha256=v011_array_sha(ids),native_prediction_array_sha256=v011_array_sha(prediction),
        columns=['id','pressure'],finite=True,native_units=True,all_rows=True,submitted=False)


def v011_run_final(project_root='.', *, resume=False, expected_selection_sha256=None,
                   expected_confirmation_file_sha256=None, expected_source_code_sha256=None):
    """Conditional final-only execution. No holdout labels are passed to any fitter."""
    root=Path(project_root).resolve(); directory=root/V011_FINAL_OUT
    v011_require(torch.cuda.is_available(),'Actual approved CUDA runtime is required')
    (root/'outputs').mkdir(exist_ok=True)
    shared=(root/'outputs/.v011_gpu.lock').open('a+')
    fcntl.flock(shared,fcntl.LOCK_EX|fcntl.LOCK_NB)
    began=time.monotonic(); previous_cwd=Path.cwd()
    try:
        os.chdir(root)
        lock,result,split,gate=v011_load_gate(root,expected_selection_sha256,
            expected_confirmation_file_sha256,expected_source_code_sha256)
        v011_load_components(root,lock['source_fingerprints'])
        seed_sequence(V011_SEED)
        prior_path=v011_verify_file(root,root/V011_PRIOR_DIR/'final_manifest.json',V011_PRIOR_MANIFEST_SHA)
        prior=json.loads(prior_path.read_text())
        v011_require(prior['status']=='complete' and prior['training_breaths']==67905
            and prior['scored_training_points']==2061922 and prior['train_sha256']==V011_TRAIN_SHA
            and prior['test_sha256']==V011_TEST_SHA and prior['sample_sha256']==V011_SAMPLE_SHA
            and prior['submission_rows']==603600 and not prior['final_refit_has_independent_holdout'],
            'Accepted all-training manifest does not describe the approved data')
        v011_verify_file(root,prior['submission_file'],V011_PRIOR_CSV_SHA)
        legacy_lock_evidence=v011_verify_legacy_lock(root,prior)
        source_fingerprints=dict(lock['source_fingerprints'],**{
            V011_FINAL_NOTEBOOK:gate['final_source_code_sha256'],'locked_before_D.json':gate['selection_sha256'],
            'confirmation_result.json':gate['confirmation_result_sha256']})
        directory.mkdir(parents=True,exist_ok=True)
        design=dict(schema=V011_FINAL_VERSION,gate=gate,winner=lock['winner'],source_fingerprints=source_fingerprints,
            train_sha256=V011_TRAIN_SHA,test_sha256=V011_TEST_SHA,sample_sha256=V011_SAMPLE_SHA,
            reused_original_manifest_sha256=V011_PRIOR_MANIFEST_SHA,reused_original_lock_evidence=legacy_lock_evidence,seed=V011_SEED,
            all_training_breaths=67905,scored_training_points=2061922,validation_used=False,
            new_submission_performed=False,official_best_preserved=.18550)
        v011_immutable_json(directory/'design.json',design,resume)
        v011_event(directory,'conditional_final_started',gate=gate,winner=lock['winner']['name'],
            gpu=torch.cuda.get_device_name(0),torch=str(torch.__version__),numpy=np.__version__,pandas=pd.__version__)
        data=v011_load_all_training(root,lock,split)
        records={}
        for member in lock['winner']['members']:
            ident=member['member_id']
            if ident in V011_ORIGINALS:
                records[ident]=v011_reuse_original(root,member,data,prior)
                v011_write_json(directory/('record_reused_'+ident+'.json'),records[ident])
                v011_event(directory,'accepted_original_reused',member_id=ident,checkpoint_sha256=records[ident]['checkpoint_sha256'])
            else:
                records[ident]=v011_fit_new_member(root,directory,member,data,gate,source_fingerprints,resume)
        all_group_sha=data['group_sha256']; all_row_sha=data['row_ids_sha256']; data_hashes=data['data_hashes']; prep_sha=_v8_statistics_sha256(data['prep'])
        del data; gc.collect(); torch.cuda.empty_cache()
        ids,test_groups,predictions,prediction_records=v011_infer_members(root,directory,lock['winner']['members'],records,source_fingerprints,resume)
        blend=v011_blend(predictions,lock['winner']['members'])
        v011_require(len(ids)==603600,'Final inference row count changed')
        blend_path=directory/'weighted_test_predictions.npy'; v011_save_array(blend_path,blend)
        row_path=directory/'test_row_ids.npy'; v011_save_array(row_path,ids)
        submission=v011_write_submission(directory,ids,blend,resume)
        # Recheck immutable prerequisites and the preserved accepted CSV after work.
        _,_,_,gate_after=v011_load_gate(root,gate['selection_sha256'],gate['confirmation_result_sha256'],gate['final_source_code_sha256'])
        v011_require(gate_after==gate,'Frozen final prerequisite changed during execution')
        v011_verify_file(root,prior['submission_file'],V011_PRIOR_CSV_SHA)
        manifest=dict(schema=V011_FINAL_VERSION,status='complete',gate=gate,design_sha256=v011_json_sha(design),
            winner=lock['winner'],members=records,weights={m['member_id']:m['weight'] for m in lock['winner']['members']},
            source_fingerprints=source_fingerprints,train_sha256=V011_TRAIN_SHA,test_sha256=V011_TEST_SHA,
            sample_sha256=V011_SAMPLE_SHA,all_training_breaths=67905,scored_training_points=2061922,
            all_training_group_sha256=all_group_sha,all_training_row_ids_sha256=all_row_sha,
            all_training_data_fingerprints=data_hashes,all_training_preprocessing_sha256=prep_sha,
            test_group_sha256=v011_array_sha(test_groups),test_row_ids_file=str(row_path),test_row_ids_file_sha256=v011_file_sha(row_path),
            prediction_members=prediction_records,weighted_prediction_file=str(blend_path),
            weighted_prediction_file_sha256=v011_file_sha(blend_path),weighted_prediction_array_sha256=v011_array_sha(blend),
            submission=submission,submission_file=submission['file'],submission_sha256=submission['file_sha256'],submission_rows=603600,
            final_fit_has_independent_holdout=False,confirmation_is_resampling_evidence=True,
            no_D_outcome_weight_search=True,official_best_preserved=.18550,old_csv_sha256=V011_PRIOR_CSV_SHA,
            submitted=False,elapsed_seconds=time.monotonic()-began,
            runtime=dict(python=platform.python_version(),torch=str(torch.__version__),numpy=np.__version__,
                pandas=pd.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
                strict_deterministic=torch.are_deterministic_algorithms_enabled()),completed_utc=datetime.now(timezone.utc).isoformat())
        v011_write_json(directory/'final_manifest.json',manifest)
        v011_event(directory,'conditional_final_complete',submission_sha256=submission['file_sha256'],elapsed_seconds=manifest['elapsed_seconds'])
        return manifest
    finally:
        os.chdir(previous_cwd)
        fcntl.flock(shared,fcntl.LOCK_UN); shared.close()




In [ ]:
# Explicit synthetic checks only. No dataset, checkpoint or D outcome is read.
def v011_final_selfcheck():
    import tempfile
    checks={}
    def check(name,value):
        if not bool(value): raise AssertionError(name)
        checks[name]=True
    def rejects(name,action):
        try: action()
        except (ValueError,TypeError,KeyError): checks[name]=True
        else: raise AssertionError(name+' did not reject')
    # A self-contained synthetic lock/outcome exercises the actual hard gates.
    cfg=dict(max_epochs=200,batch_size=256,optimizer='Adam',weight_decay=0.,loss='l1',
        scaler_phase='all',residual_baseline=False,scheduler='replay',patience=0,
        frozen_lr_schedule=[.001]*200)
    def toy_member(ident,weight):
        count=V011_ORIGINALS[ident]['epochs'] if ident in V011_ORIGINALS else 100
        return dict(member_id=ident,factory_id=ident,kind='v5' if ident in V011_ORIGINALS else 'v8',weight=weight,
            source_fingerprints={},recipe=dict(seed=V011_SEED,feature_mode='combined',
                model_spec=dict(dropout=.15),train_cfg=copy.deepcopy(cfg),loss_kind='curve_l1' if ident=='curve' else 'l1',
                selected_epochs=count,lr_schedule=cfg['frozen_lr_schedule'][:count]))
    candidates=[]
    for name,mae in [('shared_breath_offset',.191),('full_context_tcn',.195),
        ('deep_half_compact_curve_memory',.192),('deep_half_compact_curve_memory_attention',.193),
        ('deep_half_compact_curve_memory_offset',.1908)]:
        gain=.19656260982217663-mae
        candidates.append(dict(name=name,A_MAE=mae,A_gain=gain,A_interval=dict(bootstrap_95_high=-.001),eligible=gain>=.003))
    winner=dict(candidates[-1],members=[toy_member('deep',.5),toy_member('original_compact',.125),
        toy_member('curve',.125),toy_member('memory',.125),toy_member('offset',.125)])
    lock=dict(schema='confirmation_v011_locked_before_D',winner=winner,
        baseline=dict(members=[toy_member('original_compact',.5),toy_member('deep',.5)]),
        completed_A_candidates=candidates,approved_train_sha256=V011_TRAIN_SHA,source_fingerprints={},
        main_code_sha256='2'*64,original_A_validation_excluded=True,D_is_resampling_confirmation=True,
        pressure_labels_parsed_before_lock=False,D_outcomes_read_before_lock=False,
        confirmation_rule=dict(strictly_positive_gain=True,whole_breath_bootstrap_upper95_below_zero=True,
            bootstrap_seed=20261201,bootstrap_draws=1000,no_runner_up_after_failure=True,no_weight_or_epoch_search=True))
    result=dict(schema='confirmation_v011_result',status='complete',all_required_members_completed=True,
        original_A_validation_excluded=True,frozen_A_choice=True,no_runner_up_after_failure=True,
        confirmation_is_resampling_evidence=True,no_D_weight_or_epoch_selection=True,
        selection_sha256=v011_json_sha(lock),approved_train_sha256=V011_TRAIN_SHA,
        winner_name=winner['name'],source_fingerprints={},main_code_sha256='2'*64,
        decision=dict(passed=True,gain=.01,baseline_MAE=.21,candidate_MAE=.20,
            interval=dict(difference=-.01,bootstrap_95_low=-.015,bootstrap_95_high=-.005,
                seed=20261201,repeats=1000,point_count_weighted=True)))
    check('synthetic_frozen_D_pass_accepted',v011_gate_decision(lock,result,v011_json_sha(lock))==v011_json_sha(lock))
    bad=copy.deepcopy(result);bad['decision']['passed']=False
    rejects('reject_failed_D_confirmation',lambda:v011_gate_decision(lock,bad,v011_json_sha(lock)))
    bad=copy.deepcopy(result);bad['decision']['interval']['bootstrap_95_high']=0.
    rejects('reject_D_interval_touching_zero',lambda:v011_gate_decision(lock,bad,v011_json_sha(lock)))
    bad=copy.deepcopy(result);bad['decision']['gain']=.02
    rejects('reject_inconsistent_D_gain',lambda:v011_gate_decision(lock,bad,v011_json_sha(lock)))
    bad=copy.deepcopy(result);bad['winner_name']='another_candidate'
    rejects('reject_post_D_runner_up',lambda:v011_gate_decision(lock,bad,v011_json_sha(lock)))
    rejects('reject_external_selection_hash_mismatch',lambda:v011_gate_decision(lock,result,'0'*64))
    bad_lock=copy.deepcopy(lock);bad_lock['winner']['members'][0]['weight']=.6
    bad_lock['winner']['members'][1]['weight']=.025
    bad_result=copy.deepcopy(result);bad_result['selection_sha256']=v011_json_sha(bad_lock)
    rejects('reject_unapproved_frozen_weights',lambda:v011_gate_decision(bad_lock,bad_result,v011_json_sha(bad_lock)))
    bad_lock=copy.deepcopy(lock);bad_lock['winner']=dict(candidates[2],members=copy.deepcopy(winner['members']))
    bad_result=copy.deepcopy(result);bad_result['winner_name']=bad_lock['winner']['name'];bad_result['selection_sha256']=v011_json_sha(bad_lock)
    rejects('reject_nonlowest_A_choice',lambda:v011_gate_decision(bad_lock,bad_result,v011_json_sha(bad_lock)))
    with tempfile.TemporaryDirectory(prefix='v011-final-synthetic-') as temporary:
        root=Path(temporary); root.mkdir(exist_ok=True)
        ids=np.array([9,12,15,21],dtype=np.int64)
        members=[dict(member_id='deep',weight=.5),dict(member_id='offset',weight=.5)]
        predictions=dict(deep=np.array([1.,2.,3.,4.]),offset=np.array([2.,4.,6.,8.]))
        blend=v011_blend(predictions,members)
        check('fixed_weight_native_float64_all_rows',np.array_equal(blend,np.array([1.5,3.,4.5,6.])) and blend.dtype==np.float64)
        submission=v011_write_submission(root,ids,blend)
        check('synthetic_csv_exact_row_values_and_ids',submission['rows']==4 and submission['native_prediction_array_sha256']==v011_array_sha(blend))
        check('explicit_resume_preserves_csv_hash',v011_write_submission(root,ids,blend,True)['file_sha256']==submission['file_sha256'])
        rejects('reject_overwrite_without_resume',lambda:v011_write_submission(root,ids,blend))
        rejects('reject_changed_csv_values_on_resume',lambda:v011_write_submission(root,ids,blend+1.,True))
        rejects('reject_nonfinite_native_prediction',lambda:v011_blend(dict(predictions,offset=np.full(4,np.nan)),members))
        rejects('reject_prediction_shape_mismatch',lambda:v011_blend(dict(predictions,offset=np.ones(3)),members))
        rejects('reject_artifact_path_escape',lambda:v011_local(root,'../outside.csv'))
        rejects('require_external_selection_hash',lambda:v011_hex_digest(None,'selection'))
        check('compact_canonical_digest_order_invariant',v011_json_sha(dict(a=1,b=2))==v011_json_sha(dict(b=2,a=1)))
        path=root/'immutable.json';v011_immutable_json(path,dict(a=1),False)
        rejects('reject_changed_frozen_final_design',lambda:v011_immutable_json(path,dict(a=2),True))
        v011_immutable_json(path,dict(a=1),True);checks['identical_final_design_resume_allowed']=True
    return dict(schema='v011_final_synthetic_selfcheck',passed=all(checks.values()),checks=checks,
        check_count=len(checks),assignment_data_reads=0,validation_outcome_reads=0,model_fits=0,
        submission_calls=0,synthetic_only=True)




In [ ]:
V011_FINAL_RUN = os.environ.get('PA3_RUN_FINAL_V011','0')=='1'
if V011_FINAL_RUN:
    V011_FINAL_REPORT=v011_run_final(Path.cwd(),resume=os.environ.get('PA3_RESUME_FINAL_V011','0')=='1',
        expected_selection_sha256=os.environ.get('PA3_FINAL_V011_SELECTION_SHA256'),
        expected_confirmation_file_sha256=os.environ.get('PA3_FINAL_V011_D_RESULT_SHA256'),
        expected_source_code_sha256=os.environ.get('PA3_FINAL_V011_SOURCE_SHA256'))
    print(json.dumps(dict(status=V011_FINAL_REPORT['status'],submission=V011_FINAL_REPORT['submission'],submitted=False),indent=2))
